In [2]:
#CRF
!pip install sklearn-crfsuite
import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
sentences = [
 ["John", "lives", "here"], ["Mary", "works", "today"], ["Alice", "runs", "fast"],
 ["Bob", "reads", "books"], ["Paris", "is", "beautiful"], ["London", "is", "large"],
 ["Google", "hires", "people"], ["OpenAI", "builds", "models"], ["Rome", "looks", "old"],
 ["David", "likes", "music"], ["Amazon", "sells", "items"], ["Tokyo", "is", "busy"]
]

labels = [
 ["PERSON","O","O"], ["PERSON","O","O"], ["PERSON","O","O"], ["PERSON","O","O"],
 ["PLACE","O","O"], ["PLACE","O","O"], ["ORG","O","O"], ["ORG","O","O"],
 ["PLACE","O","O"], ["PERSON","O","O"], ["ORG","O","O"], ["PLACE","O","O"]
]
def word_features(sent, i):
 w = sent[i]
 return {"word": w.lower(), "is_title": w.istitle(), "prefix": w[:2].lower(),
 "suffix": w[-2:].lower(), "position0": i == 0}
X = [[word_features(s, i) for i in range(len(s))] for s in sentences]
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.25, random_state=42)
model = sklearn_crfsuite.CRF(algorithm="lbfgs", max_iterations=100)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Predicted sequences:", pred)
print("True sequences:", y_test)

C:\Users\praja\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\praja\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Predicted sequences: [['PLACE' 'O' 'O']
 ['PLACE' 'O' 'O']
 ['PLACE' 'O' 'O']]
True sequences: [['ORG', 'O', 'O'], ['PERSON', 'O', 'O'], ['PERSON', 'O', 'O']]


In [4]:
#HMM
!pip install hmmlearn scikit-learn
import numpy as np
from hmmlearn.hmm import GaussianHMM
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
rng = np.random.default_rng(42)
n = 240
X = np.zeros((n, 12, 1))
y = np.array([0]*(n//2) + [1]*(n//2))
X[:n//2, :, 0] = rng.normal(-1, 0.5, (n//2, 12))
X[n//2:, :, 0] = rng.normal(1, 0.5, (n//2, 12))
idx = rng.permutation(n); X, y = X[idx], y[idx]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {}
for cls in np.unique(y_train):
 seqs = X_train[y_train == cls]
 models[cls] = GaussianHMM(n_components=2, covariance_type="diag", n_iter=100, random_state=42)
 models[cls].fit(np.concatenate(seqs), [len(s) for s in seqs])
pred = [max(models, key=lambda c: models[c].score(seq)) for seq in X_test]
print(classification_report(y_test, pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24

    accuracy                           1.00        48
   macro avg       1.00      1.00      1.00        48
weighted avg       1.00      1.00      1.00        48

Accuracy: 1.0


In [5]:
#extreme learning machines
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train, X_test = scaler.fit_transform(X_train), scaler.transform(X_test)
rng = np.random.default_rng(42)
W = rng.normal(size=(X_train.shape[1], 100))
b = rng.normal(size=(1, 100))

H = 1 / (1 + np.exp(-np.clip(X_train @ W + b, -30, 30)))
T = np.eye(len(np.unique(y_train)))[y_train]
beta = np.linalg.pinv(H) @ T
Ht = 1 / (1 + np.exp(-np.clip(X_test @ W + b, -30, 30)))
pred = (Ht @ beta).argmax(axis=1)
print(classification_report(y_test, pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, pred))

              precision    recall  f1-score   support

           0       0.91      0.98      0.94        42
           1       0.99      0.94      0.96        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

Accuracy: 0.956140350877193


In [7]:
#capsule networks
!pip install tensorflow scikit-learn
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
X, y = load_digits(return_X_y=True)
X = (X.reshape(-1, 8, 8, 1) / 16.0).astype("float32")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
inputs = layers.Input((8, 8, 1))
x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
x = layers.Flatten()(x)
x = layers.Dense(10 * 8, activation="relu")(x)
caps = layers.Reshape((10, 8))(x)
lengths = layers.Lambda(lambda t: tf.sqrt(tf.reduce_sum(tf.square(t), axis=-1) + 1e-8))(caps)
outputs = layers.Softmax()(lengths)
model = models.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=8, batch_size=32, validation_split=0.1, verbose=1)
pred = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, pred))
model.save("capsule_style_digits.keras")


Epoch 1/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7131 - loss: 1.5696 - val_accuracy: 0.9028 - val_loss: 0.7716
Epoch 2/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9234 - loss: 0.4498 - val_accuracy: 0.9306 - val_loss: 0.3430
Epoch 3/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9389 - loss: 0.2472 - val_accuracy: 0.9514 - val_loss: 0.2307
Epoch 4/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9528 - loss: 0.1788 - val_accuracy: 0.9514 - val_loss: 0.1800
Epoch 5/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9660 - loss: 0.1427 - val_accuracy: 0.9375 - val_loss: 0.1795
Epoch 6/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9714 - loss: 0.1206 - val_accuracy: 0.9444 - val_loss: 0.1596
Epoch 7/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9675 - loss: 0.1046 - val_accuracy: 0.9514 - val_loss: 0.1529
Epoch 8/8
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9814 - loss: 0.0895 - val_accuracy: 0.9514 - val_loss: 0.134

In [10]:
#GNN
!pip install torch_geometric
import torch
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, accuracy_score
dataset = KarateClub()
data = dataset[0]

torch.manual_seed(42)
perm = torch.randperm(data.num_nodes)
train_idx, test_idx = perm[:data.num_nodes//2], perm[data.num_nodes//2:]
class GCN(torch.nn.Module):
 def __init__(self, in_channels, hidden, classes):
     super().__init__()
     self.conv1 = GCNConv(in_channels, hidden)
     self.conv2 = GCNConv(hidden, classes)
 def forward(self, x, edge_index):
     x = F.relu(self.conv1(x, edge_index))
     x = F.dropout(x, p=0.2, training=self.training)
     return self.conv2(x, edge_index)
model = GCN(dataset.num_features, 16, dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[train_idx], data.y[train_idx])
    loss.backward()
    optimizer.step()
model.eval()
with torch.no_grad():
    pred = model(data.x, data.edge_index).argmax(dim=1)[test_idx].cpu().numpy()
true = data.y[test_idx].cpu().numpy()
print(classification_report(true, pred, zero_division=0))
print("Accuracy:", accuracy_score(true, pred))
torch.save(model.state_dict(), "gcn_karate.pt")

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 11.4 MB/s eta 0:00:00
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         7
           2       1.00      1.00      1.00         1
           3       1.00      1.00      1.00         3

    accuracy                           1.00        17
   macro avg       1.00      1.00      1.00        17
weighted avg       1.00      1.00      1.00        17

Accuracy: 1.0
